In [1]:
import torch
import torch.nn.functional as F
from diffusers import DDPMPipeline, DDIMScheduler

# Looking for a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
pipe  = DDPMPipeline.from_pretrained("1aurent/ddpm-mnist")
model = pipe.unet.to(device)
model.eval()
scheduler = pipe.scheduler

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 2/2 [00:00<00:00, 36.40it/s]


In [2]:
# Looking at the scheduler configuration 
cfg = scheduler.config
print("beta_schedule       :", cfg.beta_schedule)         
print("beta_start          :", cfg.beta_start)             
print("beta_end            :", cfg.beta_end)               
print("num_train_timesteps :", cfg.num_train_timesteps)    
print("prediction_type     :", cfg.prediction_type)        

betas = scheduler.betas.cuda()
alphas = scheduler.alphas.cuda()
alphas_cumprod = scheduler.alphas_cumprod.cuda()
T_model = len(betas)

beta_schedule       : linear
beta_start          : 0.0001
beta_end            : 0.02
num_train_timesteps : 1000
prediction_type     : epsilon


In [3]:
# Log-likelihood function for the denoising problem 
def log_likelihood(x0_hat, observation, sigma_obs):
    """
    Gaussian observation model:  y ~ N(x₀, σ²_obs · I)
    log p̃(y | x̂₀) = -‖y - x̂₀‖² / (2 σ²_obs)    → (K,)
    """
    diff = (x0_hat - observation)                           # (K,1,28,28)
    return -diff.pow(2).sum(dim=(1, 2, 3)) / (2 * sigma_obs ** 2)

In [4]:
# Prediction of x0 given a denoised version xt
def predict_x0(xt, t_val):
    """
    x̂₀(xₜ) = (xₜ - √(1-ᾱₜ) · ε̂θ(xₜ,t)) / √ᾱₜ
    Requires grad on xt for the twist gradient.
    """
    ab_t       = alphas_cumprod[t_val]
    t_batch    = torch.tensor([t_val], device="cuda").expand(xt.shape[0])
    pred_noise = model(xt, t_batch).sample                  # ε̂θ  (K,1,28,28)
    x0_hat     = (xt - (1.0 - ab_t).sqrt() * pred_noise) / ab_t.sqrt()
    return x0_hat.clamp(-1.0, 1.0), pred_noise

In [5]:
def log_gaussian(x, mean, var):
    """ log N(x; mean, var·I) → (K,) """
    D = x.shape[1] * x.shape[2] * x.shape[3]
    diff = (x - mean).pow(2).sum(dim=(1, 2, 3))
    return -0.5 * (D * var.log() + diff / var)

In [17]:
def twisted_diffusion_sampler(
    model,                                 # Diffusion model
    scheduler,                             # Scheduler
    log_twisting_fn,                       # log of the twisting function 
    K,                                     # Number of particles
    device='cuda'
):
    
    alphas = scheduler.alphas.to(device)
    alphas_cumprod = scheduler.alphas_cumprod.to(device)
    betas = scheduler.betas.to(device)
    T = len(betas) 
    timesteps = list(range(T - 2, -1, -1))   # T-2 -> 0

    # Initialize x_T 
    x = torch.randn(K, 1, 28, 28, device=device)
    log_w = torch.zeros(K, device=device)
   
    for t in timesteps:
        # Resample
        w_norm = (log_w - torch.logsumexp(log_w, dim=0)).exp()
        indices = torch.multinomial(w_norm, K, replacement=True) 
        x = x[indices]
        x = x.detach().requires_grad_(True)

        # Denoise prediction
        x_hat, pred_noise = predict_x0(x, t+1)
        
        # Conditional score approximation
        s = - pred_noise/(1-alphas_cumprod[t+1]).sqrt() # Score
        log_p_tilde = log_twisting_fn(x_hat)
        grad_log_p_tilde = torch.autograd.grad(outputs=log_p_tilde.sum(), inputs=x)[0]
        s_tilde = s + grad_log_p_tilde # Approximation of the conditional score
       
        # Proposal
        mu = (1/alphas[t+1].sqrt()) * (x + betas[t+1]*s)
        mu_twisted = (1/alphas[t+1].sqrt()) * (x + betas[t+1]*s_tilde)
        noise = torch.randn_like(x)
        x = (mu_twisted + betas[t+1]*noise).clamp(-1, 1)

        # Update Weights
        with torch.no_grad():
            x_hat_next, pred_noise_next = predict_x0(x, t)
            log_p_tilde_next = log_twisting_fn(x_hat_next) 

            log_w = log_gaussian(x, mu, betas[t+1]) + log_p_tilde_next - log_gaussian(x, mu_twisted, betas[t+1]) -log_p_tilde

        x = x.detach()

    return x, w_norm

### Noisy

In [25]:
from torchvision import datasets, transforms
from torchvision.utils import save_image

transform   = transforms.Compose([transforms.ToTensor(),
                                   transforms.Normalize([0.5], [0.5])])
dataset     = datasets.MNIST("./data", train=False, download=True, transform=transform)
clean       = dataset[0][0].unsqueeze(0).cuda()

# Corrupt image
sigma_obs   = .5
noisy_obs   = (clean + sigma_obs * torch.randn_like(clean)).clamp(-1, 1)

# Define twisting function (partial application of observation)
log_twist_fn = lambda x0_hat: log_likelihood(x0_hat, noisy_obs, sigma_obs)

# Run TDS
samples, weights = twisted_diffusion_sampler(model, scheduler, log_twist_fn, K=2)

best = samples[weights.argmax()].unsqueeze(0).cpu()
save_image(torch.cat([clean.cpu(), noisy_obs.cpu(), best]),
           "images/tds_noisy_result.png", nrow=3, normalize=True)

### Inpainting

In [30]:
def create_center_mask(image_shape, mask_size=12):
    """Creates a boolean mask where the center is blocked (False = unobserved)."""
    mask = torch.ones(image_shape)
    h, w = image_shape[-2:]
    cy, cx = h // 2, w // 2
    half = mask_size // 2
    mask[..., cy-half:cy+half, cx-half:cx+half] = 0
    return mask

def log_likelihood_inpainting(x_hat, observation, mask):
    return torch.sum()

# twisting function inpainting
def log_gaussian_twisting_function_inpainting(x_hat, y_observed, mask):
    """
    Computes log p(y|x_hat) ~ N(y; x_hat * mask, variance).
    Effectively forces x_hat to match y where mask is 1.
    """
    # We only care about the error in the observed region (mask == 1)
    # y_observed should already be masked (0 in unobserved regions) or we multiply by mask
    diff = (y_observed - x_hat) * mask

    # Sum of squared errors
    sse = torch.sum(diff**2, dim=[1, 2, 3])
    return -0.5 * sse


mask = create_center_mask(clean.shape, mask_size=8).to(device)
masked_obs = clean * mask # Observed image y

# Define twisting function (partial application of observation)
log_twist_fn = lambda x0_hat: log_gaussian_twisting_function_inpainting(x0_hat, masked_obs, mask)

# Run TDS
samples, weights = twisted_diffusion_sampler(model, scheduler, log_twist_fn, K=2)

best = samples[weights.argmax()].unsqueeze(0).cpu()
save_image(torch.cat([clean.cpu(), masked_obs.cpu(), best]),
           "images/tds_inpainting_result.png", nrow=3, normalize=True)

## Trash 

In [ ]:
# Quadratic schedule of the paper - See Appendix D1-Forward Process
def make_quadratic_schedule(T=100, sigma2_min=1e-5, sigma2_max=1e-1):
    t = torch.arange(1, T + 1, dtype=torch.float64)
    sigma2              = sigma2_min + (t / T) ** 2 * sigma2_max
    alphas_cumprod      = 1.0 - sigma2
    alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]])
    betas               = (1.0 - alphas_cumprod / alphas_cumprod_prev).clamp(1e-8, 0.999)
    alphas              = 1.0 - betas
    return betas.float(), alphas.float(), alphas_cumprod.float()

betas, alphas, alphas_cumprod = make_quadratic_schedule(T=100)
betas          = betas.cuda()
alphas         = alphas.cuda()
alphas_cumprod = alphas_cumprod.cuda()

# ── DDPM posterior parameters ────────────────────────────────────────────────
# q(xₜ₋₁ | xₜ, x₀) = N(μ̃ₜ(xₜ, x₀),  β̃ₜ)
#
# β̃ₜ = βₜ · (1 - ᾱₜ₋₁) / (1 - ᾱₜ)              ← posterior variance
# μ̃ₜ = √ᾱₜ₋₁ · βₜ / (1-ᾱₜ) · x₀
#      + √αₜ  · (1-ᾱₜ₋₁) / (1-ᾱₜ) · xₜ         ← posterior mean

def ddpm_posterior(xt, x0_hat, t_val):
    """
    Compute DDPM posterior mean μ̃ₜ and variance β̃ₜ.
    All tensors: (K, 1, 28, 28)
    """
    ab_t  = alphas_cumprod[t_val]                           # ᾱₜ
    ab_tm1 = alphas_cumprod[t_val - 1] if t_val > 0 \
             else torch.tensor(1.0, device="cuda")          # ᾱₜ₋₁
    b_t   = betas[t_val]                                    # βₜ
    a_t   = alphas[t_val]                                   # αₜ

    # Posterior variance β̃ₜ (scalar)
    beta_tilde = b_t * (1.0 - ab_tm1) / (1.0 - ab_t)
    beta_tilde = beta_tilde.clamp(min=1e-8)

    # Posterior mean μ̃ₜ (K, 1, 28, 28)
    coef_x0 = ab_tm1.sqrt() * b_t / (1.0 - ab_t)
    coef_xt = a_t.sqrt()    * (1.0 - ab_tm1) / (1.0 - ab_t)
    mu_tilde = coef_x0 * x0_hat + coef_xt * xt

    return mu_tilde, beta_tilde                             # (K,1,28,28), scalar





# ── Twisted proposal step ────────────────────────────────────────────────────

def twisted_proposal_step(xt, t_val, log_twist_fn):
    """
    Sample xₜ₋₁ from the twisted proposal:

        q_twist(xₜ₋₁ | xₜ) = N(μ̃ₜ + β̃ₜ · ∇ₓₜ log p̃(y | x̂₀(xₜ)),  β̃ₜ · I)

    and return log of the proposal density evaluated at the sample,
    and log of the untwisted DDPM kernel (needed for weight update).

    Returns
    -------
    xt_prev        : (K,1,28,28) — sample from twisted proposal
    log_q_twist    : (K,)        — log q_twist(xₜ₋₁ | xₜ)
    log_p_ddpm     : (K,)        — log p_θ(xₜ₋₁ | xₜ)   (untwisted)
    log_twist_curr : (K,)        — log p̃(y | x̂₀(xₜ))
    """
    # ── 1. Compute x̂₀ and posterior parameters ────────────────────────────
    xt_in = xt.detach().requires_grad_(True)
    x0_hat, pred_noise = predict_x0(xt_in, t_val)

    # ── 2. Evaluate twisting function and take gradient wrt xₜ ────────────
    log_twist_curr = log_twist_fn(x0_hat)                   # (K,)
    grad_twist     = torch.autograd.grad(
        outputs    = log_twist_curr.sum(),
        inputs     = xt_in,
        create_graph = False,
    )[0]                                                     # (K,1,28,28)

    # Detach everything before sampling
    xt         = xt.detach()
    x0_hat     = x0_hat.detach()
    grad_twist = grad_twist.detach()

    # ── 3. DDPM posterior mean and variance ───────────────────────────────
    mu_tilde, beta_tilde = ddpm_posterior(xt, x0_hat, t_val)

    # ── 4. Shifted mean of twisted proposal ──────────────────────────────
    #   μ_twist = μ̃ₜ + β̃ₜ · ∇ₓₜ log p̃(y | x̂₀(xₜ))
    mu_twist = mu_tilde + beta_tilde * grad_twist            # (K,1,28,28)

    # ── 5. Sample xₜ₋₁ ~ N(μ_twist, β̃ₜ · I) ────────────────────────────
    eps     = torch.randn_like(xt)
    xt_prev = mu_twist + beta_tilde.sqrt() * eps             # (K,1,28,28)

    # ── 6. Log densities (for weight update) ─────────────────────────────
    D = xt.shape[1] * xt.shape[2] * xt.shape[3]             # image dimension

    def log_gaussian(x, mean, var):
        """ log N(x; mean, var·I) → (K,) """
        diff = (x - mean).pow(2).sum(dim=(1, 2, 3))
        return -0.5 * (D * var.log() + diff / var)

    log_q_twist = log_gaussian(xt_prev, mu_twist,  beta_tilde)  # twisted kernel
    log_p_ddpm  = log_gaussian(xt_prev, mu_tilde,  beta_tilde)  # untwisted kernel

    return xt_prev, log_q_twist, log_p_ddpm, log_twist_curr.detach()


# ── TDS Main Loop ────────────────────────────────────────────────────────────

def tds(
    model,
    log_twist_fn,                   # callable: (x0_hat) → (K,) log-likelihoods
    K              = 16,            # number of particles
    device         = "cuda",
):
    """
    Twisted Diffusion Sampler (TDS).

    Weight update at each step:

      log wₜ += log p̃(y|x̂₀(xₜ₋₁))          ← new twist at xₜ₋₁
              + log p_θ(xₜ₋₁|xₜ)             ← untwisted DDPM kernel
              - log q_twist(xₜ₋₁|xₜ)         ← twisted proposal (cancels shift)
              - log p̃(y|x̂₀(xₜ))              ← old twist at xₜ (cancels recursively)

    Returns
    -------
    samples : (K, 1, 28, 28)
    weights : (K,) normalized importance weights
    """
    T          = 100
    timesteps  = list(range(T - 1, -1, -1))                 # T-1 → 0
    log_w      = torch.zeros(K, device=device)
    ess_hist   = []

    # ── Initialize x_T ~ N(0, I) ─────────────────────────────────────────
    x = torch.randn(K, 1, 28, 28, device=device)

    # Initial twisting value at t=T-1
    with torch.no_grad():
        x0_init, _         = predict_x0(x, timesteps[0])
    log_twist_prev         = log_twist_fn(x0_init).detach()

    # ── Reverse loop ──────────────────────────────────────────────────────
    for t_val in timesteps:

        if t_val == 0:
            # Last step: no more transitions, just reweight
            with torch.no_grad():
                x0_hat, _ = predict_x0(x, 0)
            log_twist_curr = log_twist_fn(x0_hat).detach()
            log_w         += log_twist_curr - log_twist_prev
            break

        # 1. Twisted proposal step
        xt_prev, log_q, log_p, log_twist_curr = twisted_proposal_step(
            x, t_val, log_twist_fn
        )

        # 2. Compute twisting at xₜ₋₁ (next state)
        with torch.no_grad():
            x0_next, _     = predict_x0(xt_prev, t_val - 1)
        log_twist_next     = log_twist_fn(x0_next).detach()

        # 3. Weight update:
        #   log w += log p̃(y|x̂₀(xₜ₋₁)) + log p_θ(xₜ₋₁|xₜ)
        #          - log q_twist(xₜ₋₁|xₜ) - log p̃(y|x̂₀(xₜ))
        log_w += log_twist_next + log_p - log_q - log_twist_curr

        # 4. Normalize + ESS
        log_w_norm = log_w - torch.logsumexp(log_w, dim=0)
        w_norm     = log_w_norm.exp()
        ess        = 1.0 / w_norm.pow(2).sum().item()
        ess_hist.append(ess / K)

        # 5. Resample if ESS < K/2
        if ess < K / 2:
            idx            = systematic_resample(w_norm, K)
            xt_prev        = xt_prev[idx]
            log_twist_next = log_twist_next[idx]
            log_w          = torch.zeros(K, device=device)

        x              = xt_prev
        log_twist_prev = log_twist_next

    # Final normalized weights
    log_w_norm = log_w - torch.logsumexp(log_w, dim=0)
    return x, log_w_norm.exp(), ess_hist


def systematic_resample(weights, K):
    positions = (torch.arange(K, device=weights.device) +
                 torch.rand(1, device=weights.device)) / K
    cumsum    = weights.cumsum(dim=0)
    return torch.searchsorted(cumsum, positions).clamp(0, K - 1)